<a href="https://colab.research.google.com/github/jceltruda/Projects-in-AI-and-ML/blob/main/Project_6/Task_3/01_data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/SimRec')

# SimRec Data Processing

This notebook handles the full data preprocessing pipeline for the SimRec recommender system:

1. Download raw Amazon data files
2. Parse & filter items and reviews
3. Build ID mappings and output interaction files
4. Compute similarity scores using a sentence-transformer embedding model

## 1. Setup & Install

In [ ]:
!pip install -q sentence-transformers wget tqdm pandas numpy torch

In [ ]:
import os
import gzip
import wget
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer

## 2. Configuration

Uncomment one dataset block below to select which dataset to process.

In [ ]:
# Similarity model settings
EMBEDDING_MODEL = 'thenlper/gte-large'
SIMILARITY_BATCH_SIZE = 128
TOP_K = 1000
DELIMITER = " "

# Dataset selection: uncomment ONE block

# --- Beauty (default) ---
DATASET_NAME = "Beauty"
BASE_URL = "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/"
REVIEWS_FILE = "reviews_Beauty.json.gz"
META_FILE = "meta_Beauty.json.gz"
ITEM_FREQ_MIN = 5
REVIEWS_REMOVE_LESS_THAN = 5
preprocessing_title = lambda t: t

# # --- Tools ---
# DATASET_NAME = "Tools"
# BASE_URL = "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/"
# REVIEWS_FILE = "reviews_Tools_and_Home_Improvement.json.gz"
# META_FILE = "meta_Tools_and_Home_Improvement.json.gz"
# ITEM_FREQ_MIN = 5
# REVIEWS_REMOVE_LESS_THAN = 5
# preprocessing_title = lambda t: t

# # --- HomeKitchen ---
# DATASET_NAME = "HomeKitchen"
# BASE_URL = "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/"
# REVIEWS_FILE = "reviews_Home_and_Kitchen.json.gz"
# META_FILE = "meta_Home_and_Kitchen.json.gz"
# ITEM_FREQ_MIN = 5
# REVIEWS_REMOVE_LESS_THAN = 5
# preprocessing_title = lambda t: t

# # --- PetSupplies ---
# DATASET_NAME = "Pet"
# BASE_URL = "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/"
# REVIEWS_FILE = "reviews_Pet_Supplies.json.gz"
# META_FILE = "meta_Pet_Supplies.json.gz"
# ITEM_FREQ_MIN = 5
# REVIEWS_REMOVE_LESS_THAN = 5
# preprocessing_title = lambda t: t

# # --- Steam ---
# DATASET_NAME = "steam"
# BASE_URL = ""  # Steam requires a different download approach
# REVIEWS_FILE = ""  # Set path to your steam data
# META_FILE = ""  # Set path to your steam metadata
# ITEM_FREQ_MIN = 5
# REVIEWS_REMOVE_LESS_THAN = 5
# preprocessing_title = lambda t: t

# # --- ML-1M ---
# DATASET_NAME = "ml-1m"
# BASE_URL = ""  # ML-1M requires a different download approach
# REVIEWS_FILE = ""  # Set path to your ML-1M data
# META_FILE = ""  # Set path to your ML-1M metadata
# ITEM_FREQ_MIN = 5
# REVIEWS_REMOVE_LESS_THAN = 5
# def preprocessing_title(title):
#     title = " ".join(title.split(" ")[:-1]).strip()
#     if title.endswith(", The"):
#         title = "The " + title[:-5]
#     if title.endswith(", A"):
#         title = "A " + title[:-3]
#     return title

In [ ]:
# Create output directory for this dataset
DATA_DIR = DATASET_NAME
os.makedirs(DATA_DIR, exist_ok=True)

# Output file paths
OUT_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}.txt")
TITLES_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}-titles.txt")
ID_TO_ASIN_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}-id_to_asin.txt")
TRAIN_ITEM_FREQ_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}-train_item_freq.txt")

EMBEDDING_MODEL_SAFE = EMBEDDING_MODEL.replace('/', '_')
SIMILARITY_INDICES_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}-similarity-indices-{EMBEDDING_MODEL_SAFE}.pt")
SIMILARITY_VALUES_PATH = os.path.join(DATA_DIR, f"{DATASET_NAME}-similarity-values-{EMBEDDING_MODEL_SAFE}.pt")

print(f"Dataset: {DATASET_NAME}")
print(f"Output directory: {DATA_DIR}/")

## 3. Download Raw Data

In [ ]:
reviews_path = os.path.join(DATA_DIR, REVIEWS_FILE)
meta_path = os.path.join(DATA_DIR, META_FILE)

if BASE_URL:  # Amazon datasets
    if not os.path.exists(reviews_path):
        print(f"Downloading {REVIEWS_FILE}...")
        wget.download(BASE_URL + REVIEWS_FILE, out=reviews_path)
    else:
        print(f"{REVIEWS_FILE} already exists, skipping download.")

    if not os.path.exists(meta_path):
        print(f"Downloading {META_FILE}...")
        wget.download(BASE_URL + META_FILE, out=meta_path)
    else:
        print(f"{META_FILE} already exists, skipping download.")
else:
    print("No BASE_URL set, ensure your data files are in place manually.")

## 4. Parse & Filter Data

In [ ]:
# Load item metadata (titles)
items = dict()
skipped = 0
with gzip.open(meta_path, "r") as f:
    for line in tqdm(f, desc="Loading items"):
        json_obj = eval(line)
        asin = json_obj['asin']
        if 'title' in json_obj:
            title = json_obj['title'].replace('"', "'")
            title = title.replace("\n", " ")
            if len(title) >= 2:
                items[asin] = title
            else:
                skipped += 1
        else:
            skipped += 1

print(f"Found {len(items)} items")
print(f"Skipped {skipped} items without title")

In [ ]:
# Load reviews
reviews = defaultdict(list)
item_freq = defaultdict(int)
skipped = 0
with gzip.open(reviews_path, "r") as f:
    for line in tqdm(f, desc="Loading reviews"):
        json_obj = eval(line)
        user_id = json_obj['reviewerID']
        asin = json_obj['asin']
        timestamp = json_obj['unixReviewTime']
        if asin in items:
            reviews[user_id].append((asin, int(timestamp)))
            item_freq[asin] += 1
        else:
            skipped += 1

print(f"Found {len(reviews)} users")
print(f"Found {sum(item_freq.values())} reviews")
print(f"Skipped {skipped} item reviews without metadata")

In [ ]:
# Filter by item frequency and user review count
item_freq = {k: v for k, v in item_freq.items() if v >= ITEM_FREQ_MIN}
item_freq = dict(sorted(item_freq.items()))

removed_users_less_than = 0
removed_users_item_less_than = 0
removed_items = 0
updated_items = set()

for user_id in list(reviews.keys()):
    if len(reviews[user_id]) < REVIEWS_REMOVE_LESS_THAN:
        del reviews[user_id]
        removed_users_less_than += 1
    else:
        len_before = len(reviews[user_id])
        reviews[user_id] = [item for item in reviews[user_id] if item[0] in item_freq]
        updated_items.update([t[0] for t in reviews[user_id]])
        removed_items += len_before - len(reviews[user_id])
        if len(reviews[user_id]) <= 0:
            del reviews[user_id]
            removed_users_item_less_than += 1

print(f"Removed {removed_items} reviews of items that appear less than {ITEM_FREQ_MIN} in total")
print(f"Removed {removed_users_less_than} users with less than {REVIEWS_REMOVE_LESS_THAN} actions")
print(f"Removed {removed_users_item_less_than} users with only item count less than {REVIEWS_REMOVE_LESS_THAN}")

In [ ]:
# Recalculate item frequency & remove unused items
original_item_freq = item_freq
item_freq = defaultdict(int)
for user_id, rating_list in reviews.items():
    for item, timestamp in rating_list:
        item_freq[item] += 1

item_freq = dict(sorted(item_freq.items()))
print(f"Total of {sum(item_freq.values())} reviews")

# Remove items that no longer appear in any review
new_items = {}
new_item_freq = {}
new_original_item_freq = {}
for asin in tqdm(updated_items, desc="Cleaning items"):
    new_items[asin] = items[asin]
    new_item_freq[asin] = item_freq[asin]
    new_original_item_freq[asin] = original_item_freq[asin]

print(f"Removed {len(items) - len(new_items)} items that are not been reviewed")
item_freq = new_item_freq
items = new_items
original_item_freq = new_original_item_freq

print(f"\nItems   Reviews   Users")
print(f"{len(items):<4}   {sum(len(v) for v in reviews.values()):<7}   {len(reviews):<5}")

## 5. Build ID Mappings & Write Output Files

In [ ]:
# Create integer ID mappings
user_id_mapping = {uid: i for i, uid in enumerate(reviews.keys())}
item_id_mapping = {asin: i for i, asin in enumerate(items.keys())}

# Compute train / val / test item frequencies
train_item_freq = {k: 0 for k in item_freq.keys()}
val_item_freq = {k: 0 for k in item_freq.keys()}
test_item_freq = {k: 0 for k in item_freq.keys()}

for user_id, rating_list in reviews.items():
    sorted_list = list(map(lambda t: t[0], sorted(rating_list, key=lambda t: t[1])))
    if len(sorted_list) < 3:
        train_list = sorted_list
    else:
        train_list = sorted_list[1:-2]
        val_item_freq[sorted_list[-2]] += 1
        test_item_freq[sorted_list[-1]] += 1
    for asin in train_list:
        train_item_freq[asin] += 1

In [ ]:
# Write output files

# User-item interaction sequences (user and item IDs start from 1; 0 is reserved for padding)
with open(OUT_PATH, "w") as f:
    for user_id, rating_list in reviews.items():
        sorted_list = sorted(rating_list, key=lambda t: t[1])
        for asin, timestamp in sorted_list:
            f.write(f"{user_id_mapping[user_id] + 1} {item_id_mapping[asin] + 1}\n")

# Item ID → title mapping
with open(TITLES_PATH, "w") as f:
    for asin, title in items.items():
        f.write(f'{item_id_mapping[asin]} "{title}"\n')

# Item ID → ASIN mapping
with open(ID_TO_ASIN_PATH, "w") as f:
    for asin, item_id in item_id_mapping.items():
        f.write(f'{item_id} {asin}\n')

# Training item frequencies
with open(TRAIN_ITEM_FREQ_PATH, "w") as f:
    for asin, count in train_item_freq.items():
        f.write(f'{item_id_mapping[asin]} {count}\n')

print(f"Written: {OUT_PATH}")
print(f"Written: {TITLES_PATH}")
print(f"Written: {ID_TO_ASIN_PATH}")
print(f"Written: {TRAIN_ITEM_FREQ_PATH}")

## 6. Calculate Similarity Scores

Uses a sentence-transformer model to embed each item title, then computes
pairwise cosine similarity and retains the top-K most similar items per item.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
def sentence_transformer(model_name, batch_size, device):
    """Create a batched embedding function using a SentenceTransformer model."""
    model = SentenceTransformer(model_name, device=device)
    def embed(sentences):
        embeddings = []
        batches = [sentences[x:x+batch_size] for x in range(0, len(sentences), batch_size)]
        for batch in tqdm(batches, desc="Embedding"):
            embeddings.append(model.encode(batch, convert_to_numpy=False, convert_to_tensor=True))
        return torch.cat(embeddings, dim=0)
    return embed

embedding_func = sentence_transformer(
    model_name=EMBEDDING_MODEL,
    batch_size=SIMILARITY_BATCH_SIZE,
    device=device,
)

In [ ]:
# Load titles and item frequencies
titles_df = pd.read_csv(TITLES_PATH, names=['id', 'title'], delimiter=DELIMITER, escapechar="\\")
id_to_freq_df = pd.read_csv(TRAIN_ITEM_FREQ_PATH, names=['id', 'freq'], delimiter=DELIMITER)
id_to_freq_series = pd.Series(id_to_freq_df.freq.values, index=id_to_freq_df.id)
titles_df['freq'] = id_to_freq_series
titles_df = titles_df[['id', 'freq', 'title']]

# Apply any dataset-specific title preprocessing
titles_df['title'] = titles_df['title'].apply(np.vectorize(preprocessing_title))
print(f"Loaded {len(titles_df)} item titles")
titles_df.head()

In [ ]:
# Compute embeddings
titles_list = titles_df['title'].tolist()
titles_embeddings = embedding_func(titles_list)
print(f"Embeddings shape: {titles_embeddings.shape}")

In [ ]:
def get_similarity_matrix(embeddings, eps=1e-8, top_k=None):
    """Compute pairwise cosine similarity and optionally return only top-K per item."""
    embeddings_norm = embeddings.norm(dim=1).unsqueeze(dim=1)
    embeddings_normalized = embeddings / torch.max(embeddings_norm, eps * torch.ones_like(embeddings_norm))

    if top_k is None:
        similarity_values = embeddings_normalized @ embeddings_normalized.T
        # Fix numerical precision: ensure self-similarity is maximal
        similarity_values += torch.diag(torch.full((similarity_values.shape[0],), 1e-7, device=device))
        similarity_indices = torch.arange(similarity_values.shape[0]).unsqueeze(0).repeat(similarity_values.shape[0], 1)
    else:
        n_embeddings = embeddings.shape[0]
        chunks = max(n_embeddings // 1000, 1)
        value_list = []
        indices_list = []
        for chunk in embeddings_normalized.chunk(chunks):
            similarity_out = chunk @ embeddings_normalized.T
            values, indices = torch.topk(similarity_out, dim=-1, k=top_k, sorted=True)
            value_list.append(values)
            indices_list.append(indices)
        similarity_values = torch.cat(value_list, dim=0)
        similarity_indices = torch.cat(indices_list, dim=0)

    return similarity_values, similarity_indices

In [ ]:
similarity_values, similarity_indices = get_similarity_matrix(titles_embeddings, top_k=TOP_K)
print(f"Similarity indices shape: {similarity_indices.shape}")
print(f"Similarity values shape:  {similarity_values.shape}")

In [ ]:
# Save similarity matrices
torch.save(similarity_indices, SIMILARITY_INDICES_PATH)
torch.save(similarity_values, SIMILARITY_VALUES_PATH)
print(f"Saved: {SIMILARITY_INDICES_PATH}")
print(f"Saved: {SIMILARITY_VALUES_PATH}")